> **Public-release version:** Private animal-level inputs and source-derived observations have been removed from the notebook. The original modelling and plotting cells are retained. See the repository README for data requirements.

# NB11 — PE sensitivity analysis for normalized R, T1 and T2

**Objective.** Determine how strongly NB10 v1.2 estimates depend on two **fixed, unvalidated assumptions**: the measurement error of the selective plates and between-animal variation. Compare the direction, credible intervals and convergence of **R, T1, T2, R + T1** and **1 − T1/T2** for target and non-target mice. No manuscript changes are warranted by running this notebook.

**Relationship to existing work.** NB05 measures the observed resistance-reduction ratio among Transmid carriers (ΔT = 1 − T1/T2). NB10 instead estimates latent bacterial populations from original selective-plate CFU/mL, assuming negligible spontaneous loss of the original resistance plasmid from recipients. NB11 holds the *NB10 v1.2 Stan model, priors and input data fixed* and varies only the assumed observation and animal-heterogeneity standard deviations. It does not tune assumptions to force agreement with NB05 or a preferred biological outcome.

**PE definitions:** R = Transmid-negative recipients, assumed carbenicillin-resistant; T1 = Transmid-positive recipients retaining carbenicillin resistance; T2 = all Transmid-positive recipients; T-sensitive = T2 − T1. **No donor D occurs in PE.** Modelled plate counts: Cb = R + T1; Cb+Tc = T1; Tc = T2. Each animal is normalized against its own latent total **R + T2**, so R/(R+T2) + T2/(R+T2) = 1, while T1 is a **subset** of T2. The resistant fraction is (R+T1)/(R+T2), which is **not** equivalent to 1 − T1/T2.

**Data:** The same 32 animals and three selective-plate CFU/mL measurements as NB10 v1.2, not included in this public-release notebook. Eight animals per group. Early target and non-target are both 6 h. Later target is 16–20 h and later non-target is 24 h; **later comparisons are not time-matched**. LB counts are available for later animals only and are reserved for an independent *descriptive* check; they are not fitted. Some plates violate exact-subset relationships: measurements are retained without clipping or replacement.

**Five pre-specified refits:** baseline (plate SD 0.70; animal SD 1.80), lower/higher plate error (0.35 or 1.00), lower/higher animal heterogeneity (1.00 or 2.70). SD values are natural-log CFU units and apply equally to the three plate types or components. Each refit has separate posterior draws; do **not pool** draws across scenarios. Four chains and 1,500 warmup + 1,500 sampling draws per chain, matching NB10 v1.2. Convergence is checked separately for **every** scenario; failure makes that scenario's results provisional.

**Run sequentially in Google Colab.** NB11.01 installs dependencies; NB11.02 finds your existing `Transmid_Animal_Bayesian_Project` Google Drive folder; NB11.03 reads separately authorised PE input data (not supplied in this archive). Stan is compiled once and refitted for five independent scenarios. Outputs are saved under `06_Results/NB11_PE_Sensitivity/`, with no overwriting of previous NB files. This analysis can involve substantial sampling computation.

**Interpretation safeguards:** Good sampling diagnostics or posterior predictive coverage does *not* prove the measurement model or biology. The raw CFU/mL measurements are plate-derived estimates, not counted colonies in a calibrated Poisson likelihood. Group contrasts are not baseline-adjusted causal effects. Private KWS-derived animal data are not licensed for public redistribution.

In [ ]:
#@title NB11.01 — Install analysis packages
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q','cmdstanpy','numpy','pandas','matplotlib'])
print('NB11 dependencies installed.')


In [ ]:
#@title NB11.02 — Locate existing project and create independent output folder
from pathlib import Path
PROJECT = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    base = Path('/content/drive/MyDrive')
    direct = base / 'Transmid_Animal_Bayesian_Project'
    if direct.is_dir():
        PROJECT = direct
    else:
        hits = [p for p in base.rglob('Transmid_Animal_Bayesian_Project') if p.is_dir()]
        if len(hits) == 1:
            PROJECT = hits[0]
        elif len(hits) > 1:
            raise RuntimeError('Several matching project folders; inspect before continuing: ' + str(hits))
except (ImportError, OSError) as exc:
    print('Google Drive not used:', exc)
if PROJECT is None:
    PROJECT = Path('/content/NB11_PE_Project')
    print('Drive folder not found; outputs will remain in local Colab storage.')
DATA_DIR = PROJECT / '03_Analysis_Data' / 'NB10_PE'
MODEL_DIR = PROJECT / '05_Stan_Models'
RESULT_DIR = PROJECT / '06_Results' / 'NB11_PE_Sensitivity'
for d in (DATA_DIR, MODEL_DIR, RESULT_DIR):
    d.mkdir(parents=True,exist_ok=True)
print('Project:',PROJECT)
print('Independent results folder:',RESULT_DIR)


In [ ]:
#@title NB10.03 / NB11.03 — Load separately authorised PE culture-count data
# Public-release version: original individual-animal observations are NOT included.
# Place an authorised copy of NB10_PE_Original_CFU_derived.csv in DATA_DIR
# after obtaining the applicable data-sharing permission.
import pandas as pd, numpy as np
from IPython.display import display
CSV_NAME = 'NB10_PE_Original_CFU_derived.csv'
INPUT = DATA_DIR / CSV_NAME
if not INPUT.is_file():
    raise FileNotFoundError(f'Private PE input not provided: {INPUT}. See README for required columns.')
data = pd.read_csv(INPUT, dtype={'animal_id':str})
GROUPS = ['Early — Target', 'Early — Non-target',
          'Later — Target', 'Later — Non-target']
COLS = ['Cb_CFU_ml', 'CbTc_CFU_ml', 'Tc_CFU_ml']
required = {'animal_id','group','hours_post_infection','LB_CFU_ml',*COLS}
assert required.issubset(data.columns), f'Missing PE columns: {required-set(data.columns)}'
assert len(data) == 32 and data.animal_id.nunique() == 32
assert set(data.group) == set(GROUPS)
assert all((data.group == g).sum() == 8 for g in GROUPS)
assert np.isfinite(data[COLS].values.astype(float)).all()
assert (data[COLS] > 0).all().all()
assert data[data.group.eq('Early — Target')].hours_post_infection.eq(6).all()
assert data[data.group.eq('Early — Non-target')].hours_post_infection.eq(6).all()
assert data[data.group.eq('Later — Target')].hours_post_infection.value_counts().to_dict() == {20:7,16:1}
assert data[data.group.eq('Later — Non-target')].hours_post_infection.eq(24).all()
assert data[data.group.str.startswith('Early')].LB_CFU_ml.isna().all()
assert data[data.group.str.startswith('Later')].LB_CFU_ml.notna().all()
print('PE input validated; 32 animals (private source supplied separately).')


In [ ]:
#@title NB11.04 — Audit plate relationships and lock pre-specified scenarios
# In the model, Cb+Tc is a biological subset of Tc, and of Cb.
# Input measurements are noisy, and are never clipped or treated as exact latent counts.
audit = data[['animal_id','group','hours_post_infection',*COLS,'LB_CFU_ml']].copy()
audit['CbTc_greater_Tc'] = audit.CbTc_CFU_ml > audit.Tc_CFU_ml
audit['CbTc_greater_Cb'] = audit.CbTc_CFU_ml > audit.Cb_CFU_ml
audit['Cb_greater_LB'] = audit.Cb_CFU_ml > audit.LB_CFU_ml
audit['Tc_greater_LB'] = audit.Tc_CFU_ml > audit.LB_CFU_ml
audit.to_csv(RESULT_DIR/'NB11_input_audit.csv',index=False)
print('Input data check:')
for field in ['CbTc_greater_Tc','CbTc_greater_Cb','Cb_greater_LB','Tc_greater_LB']:
    print(f'  {field}: {int(audit[field].sum())} animals')

# Pre-specified values; do not pick a scenario because its result seems more plausible.
SCENARIOS = [
    {'id':'S0_baseline', 'plate_sd':0.70, 'animal_sd':1.80},
    {'id':'S1_plate_lower', 'plate_sd':0.35, 'animal_sd':1.80},
    {'id':'S2_plate_higher','plate_sd':1.00, 'animal_sd':1.80},
    {'id':'S3_animal_lower','plate_sd':0.70, 'animal_sd':1.00},
    {'id':'S4_animal_higher','plate_sd':0.70, 'animal_sd':2.70},
]
scenario_table = pd.DataFrame(SCENARIOS)
assert scenario_table.id.is_unique and (scenario_table[['plate_sd','animal_sd']]>0).all().all()
scenario_table.to_csv(RESULT_DIR/'NB11_prespecified_scenarios.csv',index=False)
print('Five independent scenarios (natural-log CFU/mL SD):')
display(scenario_table)


In [ ]:
#@title NB11.05 — Compile the unchanged NB10 v1.2 Stan model
from cmdstanpy import CmdStanModel, cmdstan_path, install_cmdstan
import tempfile, hashlib
try:
    print('CmdStan:',cmdstan_path())
except ValueError:
    print('CmdStan missing; installing in Colab runtime.')
    if not install_cmdstan(cores=2,progress=True):
        raise RuntimeError('CmdStan installation failed')
# Identical likelihood, priors and parameterization to NB10 v1.2.
STAN_CODE = r'''// NB10 v1.2: nonnegative R, T1, T-sensitive abundances; no donors in PE.
// Data are workbook CFU/mL estimates, not raw colony counts.
data {
  int<lower=1> N;
  int<lower=2> G;
  array[N] int<lower=1,upper=G> group;
  matrix<lower=0>[N,3] y;  // Cb, Cb+Tc, Tc
  vector<lower=0>[3] sd_plate;   // fixed, not estimated
  vector<lower=0>[3] sd_animal;  // fixed, not estimated
}
parameters {
  matrix[G,3] mu_log_population;  // per-group log CFU/mL means of R,T1,Ts
  matrix[N,3] z;                  // noncentred animal deviations
}
transformed parameters {
  matrix[N,3] log_population;
  for (i in 1:N)
    for (k in 1:3)
      log_population[i,k] = mu_log_population[group[i],k] + sd_animal[k]*z[i,k];
}
model {
  to_vector(mu_log_population) ~ normal(15,5);  // weakly informative proper prior
  to_vector(z) ~ std_normal();
  for (i in 1:N) {
    log(y[i,1]) ~ student_t(4,
       log_sum_exp(log_population[i,1], log_population[i,2]), sd_plate[1]);
    log(y[i,2]) ~ student_t(4, log_population[i,2], sd_plate[2]);
    log(y[i,3]) ~ student_t(4,
       log_sum_exp(log_population[i,2], log_population[i,3]), sd_plate[3]);
  }
}
generated quantities {
  matrix[N,3] log_y_rep;
  for (i in 1:N) {
    log_y_rep[i,1] = student_t_rng(4,
       log_sum_exp(log_population[i,1],log_population[i,2]), sd_plate[1]);
    log_y_rep[i,2] = student_t_rng(4,log_population[i,2],sd_plate[2]);
    log_y_rep[i,3] = student_t_rng(4,
       log_sum_exp(log_population[i,2],log_population[i,3]),sd_plate[3]);
  }
}
'''
MODEL_SOURCE = MODEL_DIR / 'NB11_PE_sensitivity_same_NB10_v1_2_model.stan'
if MODEL_SOURCE.exists() and MODEL_SOURCE.read_text() != STAN_CODE:
    raise RuntimeError('Existing Stan file differs; refusing to overwrite')
if not MODEL_SOURCE.exists():
    MODEL_SOURCE.write_text(STAN_CODE)
RUNTIME_DIR = Path(tempfile.mkdtemp(prefix='nb11_pe_'))
STAN_LOCAL = RUNTIME_DIR / MODEL_SOURCE.name
STAN_LOCAL.write_text(STAN_CODE)
model = CmdStanModel(stan_file=str(STAN_LOCAL))
STAN_SHA = hashlib.sha256(STAN_CODE.encode()).hexdigest()
print('Verified Stan source SHA256:',STAN_SHA)
print('Compiled once:',STAN_LOCAL)


In [ ]:
#@title NB11.06 — Refit all scenarios and calculate convergence checks
# Four chains; independent fits using identical Stan code, input records, priors,
# number of iterations and base RNG seed. Only two fixed SD assumptions vary.
import json
from datetime import datetime, timezone
GROUP_INDEX = [GROUPS.index(g)+1 for g in data.group]
Y = data[COLS].to_numpy(dtype=float)
N = len(data)
DIAG_ROWS = []
FITS = {}
for j, s in enumerate(SCENARIOS):
    sid = s['id']
    print(f'\n=== NB11 scenario {j+1}/{len(SCENARIOS)}: {sid} '
          f'(plate SD {s["plate_sd"]}, animal SD {s["animal_sd"]}) ===')
    chain_dir = RUNTIME_DIR / sid / 'chains'
    chain_dir.mkdir(parents=True,exist_ok=True)
    stan_input = {
        'N':N,'G':len(GROUPS),'group':GROUP_INDEX,'y':Y,
        'sd_plate':[s['plate_sd']]*3,
        'sd_animal':[s['animal_sd']]*3,
    }
    try:
        fit = model.sample(data=stan_input,
            chains=4,parallel_chains=2,
            iter_warmup=1500,iter_sampling=1500,
            seed=20261125,adapt_delta=0.95,max_treedepth=12,
            output_dir=str(chain_dir), show_progress=True)
        summary = fit.summary()
        focus = summary.index.to_series().str.startswith(
            ('mu_log_population[','z[','log_population['))
        ps = summary.loc[focus]
        max_rhat = float(ps.R_hat.max())
        min_ess = float(ps.ESS_bulk.min())
        methods = fit.method_variables()
        diverge = int(np.sum(methods['divergent__']))
        treedepth = methods['treedepth__']
        hits = int(np.sum(treedepth >= 12))
        ok = (np.isfinite(max_rhat) and np.isfinite(min_ess)
              and max_rhat < 1.01 and min_ess >= 400
              and diverge == 0 and hits == 0)
        ps[['Mean','StdDev','ESS_bulk','R_hat']].to_csv(
            RESULT_DIR / f'{sid}_stan_diagnostics.csv')
        FITS[sid] = fit
        rec = {**s,'fit_completed':True,'convergence_pass':bool(ok),
               'max_rhat':max_rhat,'min_bulk_ess':min_ess,
               'divergences':diverge,'treedepth_hits':hits,'error':''}
        print('Convergence', 'PASS' if ok else 'FAIL / provisional',
              f'R-hat {max_rhat:.4f}, ESS {min_ess:.0f},'
              f' divergences {diverge}, treedepth hits {hits}')
    except Exception as exc:
        rec = {**s,'fit_completed':False,'convergence_pass':False,
               'max_rhat':np.nan,'min_bulk_ess':np.nan,
               'divergences':np.nan,'treedepth_hits':np.nan,
               'error':f'{type(exc).__name__}: {exc}'}
        print('Scenario failed:',rec['error'])
    DIAG_ROWS.append(rec)
    pd.DataFrame(DIAG_ROWS).to_csv(RESULT_DIR/'NB11_scenario_convergence.csv',index=False)
convergence = pd.DataFrame(DIAG_ROWS)
display(convergence)
if not convergence.convergence_pass.all():
    print('WARNING: one or more scenarios did not pass convergence. '
          'Do not treat their estimates as validated.')
print('Fit processing complete; results preserved per scenario.')


In [ ]:
#@title NB11.07 — Extract R, T1, T2 and resistance-loss posteriors
# Every quantity is computed for each posterior draw and animal BEFORE group averaging.
# This prevents taking a ratio of group medians in place of the desired animal ratio.
def qs(x):
    return tuple(float(v) for v in np.quantile(x,[.025,.5,.975]))

GROUP_ROWS=[]
CONTRAST_ROWS=[]
PPC_ROWS=[]
LB_ROWS=[]

for scenario in SCENARIOS:
    sid=scenario['id']
    if sid not in FITS:
        continue
    fit=FITS[sid]
    passed=bool(convergence.loc[convergence.id.eq(sid),'convergence_pass'].iloc[0])
    logs=np.asarray(fit.stan_variable('log_population'))
    assert logs.shape[1:] == (N,3), (sid,logs.shape)
    pops=np.exp(logs)
    R=pops[:,:,0]; T1=pops[:,:,1]; Ts=pops[:,:,2]
    T2=T1+Ts; total=R+T2; resistant=R+T1
    vals={
        'R_CFU_ml':R, 'T1_CFU_ml':T1,'T2_CFU_ml':T2,
        'resistant_R_plus_T1_CFU_ml':resistant,
        'R_fraction':R/total,'T1_fraction':T1/total,
        'T2_fraction':T2/total,
        'resistant_fraction':resistant/total,
        'loss_among_T2':Ts/T2,
    }
    assert np.allclose(vals['R_fraction']+vals['T2_fraction'],1.0)
    assert np.all(vals['T1_fraction'] <= vals['T2_fraction']+1e-12)
    # posterior group-level means for fractions, medians across animals for CFU/mL
    groupdraws={}
    for g in GROUPS:
        mask=(data.group==g).to_numpy()
        for key,v in vals.items():
            agg = 'median_across_animals' if key.endswith('_CFU_ml') else 'mean_across_animals'
            draws = np.median(v[:,mask],axis=1) if agg.startswith('median') else v[:,mask].mean(axis=1)
            groupdraws[(g,key)] = draws
            lo,mid,hi=qs(draws)
            GROUP_ROWS.append(dict(scenario=sid,convergence_pass=passed,group=g,
                quantity=key,aggregation=agg,q025=lo,median=mid,q975=hi))
    for ep,tar,non,matched in [
        ('Early 6 h','Early — Target','Early — Non-target',True),
        ('Later (unmatched)','Later — Target','Later — Non-target',False)]:
        for key in vals:
            t=groupdraws[(tar,key)];n=groupdraws[(non,key)]
            diff=t-n
            ratio=t/n
            qd=qs(diff);qr=qs(ratio)
            CONTRAST_ROWS.append(dict(scenario=sid,convergence_pass=passed,
                endpoint=ep,time_matched=matched,quantity=key,
                target_minus_nontarget_q025=qd[0],
                target_minus_nontarget_median=qd[1],
                target_minus_nontarget_q975=qd[2],
                target_over_nontarget_q025=qr[0],
                target_over_nontarget_median=qr[1],
                target_over_nontarget_q975=qr[2],
                posterior_P_target_lower=float((diff<0).mean())))
    # In-sample PPC: observational log plate counts compared to replicated plate data.
    reps=np.asarray(fit.stan_variable('log_y_rep'))
    assert reps.shape[1:] == (N,3),reps.shape
    for i,animal in data.iterrows():
        for j,col in enumerate(COLS):
            observed_log=np.log(float(animal[col]))
            lo,med,hi=qs(reps[:,i,j])
            PPC_ROWS.append(dict(scenario=sid,convergence_pass=passed,
                animal_id=animal.animal_id,group=animal.group,
                plate=col,observed_log=observed_log,
                predicted_log_q025=lo,predicted_log_median=med,
                predicted_log_q975=hi,
                outside_95=bool(observed_log<lo or observed_log>hi)))
        if np.isfinite(animal.LB_CFU_ml):
            obs=float(animal.LB_CFU_ml)
            lo,med,hi=qs(total[:,i])
            LB_ROWS.append(dict(scenario=sid,convergence_pass=passed,
                animal_id=animal.animal_id,group=animal.group,
                observed_LB_CFU_ml=obs,latent_total_q025=lo,
                latent_total_median=med,latent_total_q975=hi,
                outside_latent_95=bool(obs<lo or obs>hi)))
    print(f'Extracted {sid}:', 'PASS' if passed else 'PROVISIONAL')

by_group=pd.DataFrame(GROUP_ROWS)
contrasts=pd.DataFrame(CONTRAST_ROWS)
ppc=pd.DataFrame(PPC_ROWS)
lb=pd.DataFrame(LB_ROWS)
for filename,df in [
    ('NB11_group_posterior_summaries.csv',by_group),
    ('NB11_target_nontarget_contrasts.csv',contrasts),
    ('NB11_selective_plate_PPC.csv',ppc),
    ('NB11_later_LB_check.csv',lb)]:
    df.to_csv(RESULT_DIR/filename,index=False)
print('Group medians (fractions are 0–1, not percentages):')
display(by_group[by_group.quantity.isin(['R_fraction','T1_fraction','T2_fraction',
                                      'resistant_fraction','loss_among_T2'])][
    ['scenario','convergence_pass','group','quantity','q025','median','q975']].round(4))


In [ ]:
#@title NB11.08 — Compare assumptions, predictive checks and primary outcomes
# Separate converged and nonconverged fits. All outputs retain a validation flag.
ppc_summary=(ppc.groupby('scenario',as_index=False).agg(
    n_predictive=('outside_95','size'),
    n_outside_95=('outside_95','sum')) if not ppc.empty else pd.DataFrame())
lb_summary=(lb.groupby('scenario',as_index=False).agg(
    n_LB=('outside_latent_95','size'),
    n_LB_outside_95=('outside_latent_95','sum')) if not lb.empty else pd.DataFrame())
quality=(convergence.merge(ppc_summary.rename(columns={'scenario':'id'}),on='id',how='left')
         .merge(lb_summary.rename(columns={'scenario':'id'}),on='id',how='left'))
quality.to_csv(RESULT_DIR/'NB11_quality_and_predictive_checks.csv',index=False)
print('Fit convergence and predictive checks:')
display(quality)
print('Important: in-sample PPC is not independent validation; LB was not fitted.')

# Key estimates target vs non-target: early endpoint is time-matched.
focus=['R_fraction','T1_fraction','T2_fraction','resistant_fraction',
       'resistant_R_plus_T1_CFU_ml','R_CFU_ml','loss_among_T2']
key=contrasts[(contrasts.endpoint=='Early 6 h') & contrasts.quantity.isin(focus)]
key.to_csv(RESULT_DIR/'NB11_primary_6h_contrasts.csv',index=False)
print('Primary 6-hour target minus non-target contrasts:')
display(key[['scenario','convergence_pass','quantity',
     'target_minus_nontarget_q025','target_minus_nontarget_median',
     'target_minus_nontarget_q975','posterior_P_target_lower']].round(4))

# The non-target later observation is a useful diagnostic of sensitivity to assumptions,
# NOT an estimate of the mechanistic off-target cutting rate.
non_target_later=by_group[(by_group.group=='Later — Non-target') &
                          (by_group.quantity=='loss_among_T2')].copy()
print('Non-target later: posterior group mean of 1 − T1/T2:')
display(non_target_later[['scenario','convergence_pass','q025','median','q975']].round(4))
non_target_later.to_csv(RESULT_DIR/'NB11_non_target_later_T2_resistance_loss.csv',index=False)


In [ ]:
#@title NB11.09 — Plot sensitivity of resistant fraction and within-T2 loss
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(13,5))
labels=[s['id'].replace('_',' ') for s in SCENARIOS]
sids=[s['id'] for s in SCENARIOS]
# Plot target–non-target differences in total resistant fraction at the 6h matched endpoint.
ax=axes[0]
f=contrasts[(contrasts.endpoint=='Early 6 h') &
            (contrasts.quantity=='resistant_fraction')].set_index('scenario')
for j,sid in enumerate(sids):
    if sid not in f.index:continue
    row=f.loc[sid]; y=row.target_minus_nontarget_median
    ax.errorbar(y,j,xerr=[[y-row.target_minus_nontarget_q025],
                          [row.target_minus_nontarget_q975-y]],
                fmt='o',capsize=3)
    if not row.convergence_pass:
        ax.annotate('convergence failed',(y,j),xytext=(4,6),textcoords='offset points',fontsize=8)
ax.axvline(0,ls='--',lw=1)
ax.set_title('6 h: target − non-target resistant fraction')
ax.set_xlabel('Difference in fraction (R + T1)/(R + T2)')
ax.set_yticks(range(len(sids)),labels)
ax.invert_yaxis();ax.grid(axis='x',alpha=.25)

ax=axes[1]
f=by_group[(by_group.group=='Later — Non-target') &
           (by_group.quantity=='loss_among_T2')].set_index('scenario')
for j,sid in enumerate(sids):
    if sid not in f.index:continue
    row=f.loc[sid];y=row['median']
    ax.errorbar(y,j,xerr=[[y-row.q025],[row.q975-y]],fmt='o',capsize=3)
    if not row.convergence_pass:
        ax.annotate('convergence failed',(y,j),xytext=(4,6),textcoords='offset points',fontsize=8)
ax.set_xlim(0,1)
ax.set_title('Later non-target: 1 − T1/T2 (exploratory)')
ax.set_xlabel('Mean within-T2 resistance-loss fraction')
ax.set_yticks(range(len(sids)),labels)
ax.invert_yaxis();ax.grid(axis='x',alpha=.25)
fig.suptitle('NB11 PE sensitivity (posterior medians and 95% credible intervals)')
fig.tight_layout()
fig.savefig(RESULT_DIR/'NB11_PE_sensitivity_critical_outcomes.png',dpi=180)
plt.show()


In [ ]:
#@title NB11.10 — Write a self-contained audit summary
from datetime import datetime, timezone
summary_lines=[
    'NB11 — PE sensitivity of NB10 v1.2 population model',
    'UTC run: '+datetime.now(timezone.utc).isoformat(),
    'Original PE source: NMS003_Part 6_ client data_CFUs (3).xlsx',
    'Source-derived CSV SHA256: '+CSV_SHA,
    'Stan source SHA256: '+STAN_SHA,
    '32 distinct PE animals: 8 per target status by early/later endpoint',
    'No donor bacteria; R Transmid-negative, T1 resistant Transmid-positive,',
    'T2 all Transmid-positive; T-sensitive = T2 − T1.',
    'Spontaneous resistance-plasmid loss in R assumed zero, not proven.',
    'Selective plate CFU/mL are observed summaries with imperfect measurement.',
    'Five sensitivity scenarios change fixed plate or animal log-SD only.',
    'All other Stan model terms and priors match NB10 v1.2.',
    'Stan chains: 4; warmup: 1500; samples: 1500 per chain; seed 20261125.',
    'Target/non-target early sample time: 6h each; matched endpoint.',
    'Later target: 16–20h; later non-target: 24h; NOT time matched.',
    'Mean fractions are per-animal normalized before group aggregation.',
    'T1 subset T2; R+T2 total; R+T1 resistant.',
    'All group contrasts exploratory, not baseline-adjusted causal effects.',
    'Convergence is required but not proof of measurement validity.',
    'Posterior predictive intervals are in-sample. LB is descriptive only.',
    'No NB05, NB09, NB10 v1.2, manuscript, supplement or GitHub changes.',
    'Private KWS-derived datasets are not authorized for publication.',
    '',
    'Scenario diagnostics:'
]
for rec in DIAG_ROWS:
    summary_lines.append(str(rec))
summary_txt='\n'.join(summary_lines)+'\n'
(RESULT_DIR/'NB11_analysis_summary.txt').write_text(summary_txt)
print(summary_txt)
print('All result files are in:',RESULT_DIR)
print('Upload the NB11_PE_Sensitivity results folder as a ZIP for scientific review.')
